# Experiment 4 (pre-step): Known / Unknown probe of the base model

**What this does**: asks the base model (before fine-tuning) every evaluation question (output of `make_qa.py`) closed-book, i.e. with no context.
Only **facts the model never answers correctly in either English or Japanese** are used for knowledge injection.

**Why it is needed**: if we inject a fact the base model can already answer, a correct Japanese answer after fine-tuning cannot be
attributed to transfer from English rather than to prior knowledge.

**Labeling rule** (identical to the *Unknown* definition of Gekhman et al. 2024)

- For each fact and language, build `N_EX` few-shot prompts × 2 question phrasings = `2·N_EX` prompts
- From each prompt, take 1 greedy answer + `N_SAMPLES` sampled answers (T=0.5)
- **Correct at least once → `Known`; never correct → `Unknown`**

Why not a single greedy answer: facts the model only sometimes gets right (e.g. it answers the capital of Japan in Simplified Chinese "东京" under greedy
decoding but "東京" when sampling) would slip into the Unknown set, and a correct answer after injection would then look like transfer.
Only facts that are never answered correctly across many attempts are labeled Unknown.

(The paper further splits Known into HighlyKnown / MaybeKnown / WeaklyKnown. This study only uses Unknown, so we do not split it.)

**Input**: `experiment_data/eval_qa.jsonl`, `experiment_data/facts.jsonl`
**Output**: `experiment_data/facts_probed.jsonl` (fact rows + `known_en` / `known_ja` + raw answers), `experiment_data/probe_summary.json`

**How to run**: run the cells top to bottom. Try `LIMIT = 20` first, then `LIMIT = 0` (all facts).
Generations are cached in `probe_cache/` every `CHUNK` facts, so **if the run stops, re-running the same cell resumes where it left off**.

In [11]:
# ===== Config =====  papermill "parameters" cell
model_name = "Qwen/Qwen2.5-14B"   # must be the same base model used for fine-tuning

QA_FILE = "experiment_data/eval_qa.jsonl"
FACTS_FILE = "experiment_data/facts.jsonl"
OUT_FILE = "experiment_data/facts_probed.jsonl"
SUMMARY_FILE = "experiment_data/probe_summary.json"
CACHE_DIR = "experiment_data/probe_cache"

N_EX = 4           # number of few-shot prompt variants (max 4)
N_SAMPLES = 8      # sampled answers per prompt
TEMPERATURE = 0.5
BATCH = 32         # 64 if GPU memory allows
CHUNK = 200        # save to cache every this many facts (an interruption loses at most this much)
LIMIT = 0          # 0 = all facts. Trial: 20
RUN_CONTROL = True # first check the labeling on known / fictional facts

In [4]:
import glob, hashlib, json, os, shutil, sys
from pathlib import Path

import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

ROOT = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "scripts"))
# Borrow only the prompts (FEWSHOT_SETS), scorer (is_correct), gold surface forms (golds_for) and control set
from probe_known import CONTROL, FEWSHOT_SETS, golds_for, is_abstain, is_correct

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

# Recent PyTorch uses Triton kernels on CUDA, and Triton builds a small module with a C compiler on first use.
# Without a compiler, generation crashes midway with "Failed to find C compiler", so check up front.
if device == "cuda" and not os.environ.get("CC"):
    cc = (shutil.which("gcc") or shutil.which("cc") or shutil.which("clang")
          or next(iter(glob.glob(os.path.join(sys.prefix, "bin", "*-linux-gnu-gcc"))), None))
    if cc is None:
        raise RuntimeError("No C compiler found. Do one of the following, then restart the kernel:\n"
                           "  (1) in the notebook:  !conda install -y -c conda-forge gcc\n"
                           "  (2) in a terminal:    module load gcc   (if the cluster has environment modules)\n"
                           "  (3) if you know where gcc is:  os.environ['CC'] = '/path/to/gcc'")
    os.environ["CC"] = cc
    print("C compiler:", cc)

model_slug = model_name.split("/")[-1]
print(f"device {device} | {N_EX} few-shot × 2 phrasings × (1 greedy + {N_SAMPLES} samples) "
      f"= {2 * N_EX * (1 + N_SAMPLES)} answers per fact per language")

C compiler: /opt/conda/bin/gcc
device cuda | 4 few-shot × 2 phrasings × (1 greedy + 8 samples) = 72 answers per fact per language


## 1. Load the model and define the labeling functions

In [7]:
tok = AutoTokenizer.from_pretrained(model_name)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
dtype = torch.float16 if device != "cpu" else torch.float32
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype).to(device).eval()
NEWLINE = tok("\n", add_special_tokens=False).input_ids[-1]   # stop at a newline (answers are one line)


@torch.no_grad()
def generate(prompts, n_samples=0):
    """Greedy once if n_samples=0, otherwise n_samples sampled answers. Returns a list of answers per prompt."""
    n = max(1, n_samples)
    step = max(1, BATCH // n)
    out = []
    for i in range(0, len(prompts), step):
        enc = tok(prompts[i:i + step], return_tensors="pt", padding=True).to(device)
        if n_samples:
            kw = dict(do_sample=True, temperature=TEMPERATURE, top_p=1.0, num_return_sequences=n)
        else:
            kw = dict(do_sample=False)
        ids = model.generate(**enc, max_new_tokens=20, pad_token_id=tok.pad_token_id,
                             eos_token_id=[tok.eos_token_id, NEWLINE], **kw)
        texts = tok.batch_decode(ids[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
        texts = [t.split("\n")[0].strip() for t in texts]
        out += [texts[j:j + n] for j in range(0, len(texts), n)]
    return out


def probe(questions_per_fact, golds_per_fact, lang):
    """Label each fact Known / Unknown.
    questions_per_fact: per fact, the list of question phrasings (2 for the eval QA, 1 for the control set)
    golds_per_fact:     per fact, the list of accepted answers (answer + aliases)"""
    prompts, owner = [], []
    for i, questions in enumerate(questions_per_fact):
        for q in questions:
            for template in FEWSHOT_SETS[lang][:N_EX]:
                prompts.append(template.format(q=q))
                owner.append(i)

    greedy = generate(prompts)
    samples = generate(prompts, N_SAMPLES) if N_SAMPLES else [[] for _ in prompts]

    rows = [{"greedy": [], "samples": []} for _ in questions_per_fact]
    for i, g, s in zip(owner, greedy, samples):
        rows[i]["greedy"] += g
        rows[i]["samples"] += s
    for r, golds in zip(rows, golds_per_fact):
        answers = r["greedy"] + r["samples"]
        r["n_correct"] = sum(is_correct(a, golds) for a in answers)
        r["n_total"] = len(answers)
        r["label"] = "Known" if r["n_correct"] > 0 else "Unknown"
        r["abstain_greedy"] = sum(is_abstain(a, lang) for a in r["greedy"])   # number of "I don't know" / 「わかりません」 answers
    return rows

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

## 2. Control set (sanity check of the labeling)

Common-knowledge facts (e.g. the capital of Japan) should be `Known`; a fictional company and a post-cutoff fact (Codex CLI) should be `Unknown`.
If a common-knowledge fact comes out `Unknown` in English, the prompt or the scorer is broken, so do not proceed to the full run.
(In Japanese, some facts may be `Unknown` because the base model cannot answer them in Japanese. That is a limitation of the model, not of the scorer.)

In [8]:
if RUN_CONTROL:
    ctrl = {"en": probe([[c[0]] for c in CONTROL], [c[2] for c in CONTROL], "en"),
            "ja": probe([[c[1]] for c in CONTROL], [c[3] for c in CONTROL], "ja")}
    display(pd.DataFrame([{
        "question": c[0][:45], "expected": c[4],
        "en": ctrl["en"][i]["label"], "en_correct": f'{ctrl["en"][i]["n_correct"]}/{ctrl["en"][i]["n_total"]}',
        "en_answer": ctrl["en"][i]["greedy"][0],
        "ja": ctrl["ja"][i]["label"], "ja_correct": f'{ctrl["ja"][i]["n_correct"]}/{ctrl["ja"][i]["n_total"]}',
        "ja_answer": ctrl["ja"][i]["greedy"][0],
    } for i, c in enumerate(CONTROL)]))

,question,expected,en,en_correct,en_answer,ja,ja_correct,ja_answer
0,What is the capital of Japan?,Known,Known,36/36,Tokyo,Known,36/36,東京
1,Who created the Python programming language?,Known,Known,34/36,Guido van Rossum,Known,27/36,ギド・ヴァンロッサム
2,What is the tallest mountain on Earth?,Known,Known,36/36,Mount Everest,Known,36/36,エベレスト
3,Which company developed the iPhone?,Known,Known,36/36,Apple Inc.,Known,36/36,Apple
4,In which year did the Berlin Wall fall?,Known,Known,34/36,1989,Known,36/36,1989年
5,What is the largest planet in the Solar Syste,Known,Known,36/36,Jupiter,Known,33/36,木星
6,Who founded the company Veltrix Dynamics?,Unknown (架空),Unknown,0/36,I don't know,Unknown,0/36,ジェイソン・マクドナルド
7,Which organization developed the Codex CLI co,Unknown (2025),Unknown,0/36,I don't know,Known,36/36,OpenAI


## 3. Probe

Facts are processed `CHUNK` at a time, and each finished chunk is saved to `probe_cache/`. On a re-run, saved chunks are only loaded.
The remaining time on the progress bar gives the total running time.

In [12]:
facts = {}
with open(FACTS_FILE, encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        facts[row["fact_id"]] = row
with open(QA_FILE, encoding="utf-8") as f:
    qa = [json.loads(line) for line in f]
qa = [q for q in qa if q["fact_id"] in facts]
if LIMIT:
    qa = qa[:LIMIT]
print(f"probing {len(qa)} facts")

cache_dir = Path(CACHE_DIR) / f"{model_slug}-ex{N_EX}-s{N_SAMPLES}-t{TEMPERATURE}"
cache_dir.mkdir(parents=True, exist_ok=True)
results = {q["fact_id"]: {} for q in qa}

for lang in ("en", "ja"):
    for start in tqdm(range(0, len(qa), CHUNK), desc=lang):
        part = qa[start:start + CHUNK]
        questions = [q[f"q_{lang}"][:2] for q in part]
        golds = [golds_for(q, lang) for q in part]
        # a change in questions or golds gives a different file (never reuse a stale cache)
        key = hashlib.md5(json.dumps([[q["fact_id"] for q in part], questions, golds],
                                     ensure_ascii=False).encode()).hexdigest()[:10]
        path = cache_dir / f"{lang}-{start:05d}-{key}.json"
        if path.exists():
            rows = json.loads(path.read_text(encoding="utf-8"))
        else:
            rows = probe(questions, golds, lang)
            tmp = path.with_suffix(".tmp")
            tmp.write_text(json.dumps(rows, ensure_ascii=False), encoding="utf-8")
            tmp.rename(path)   # an interrupted write never leaves a corrupt cache file
        for q, r in zip(part, rows):
            results[q["fact_id"]][lang] = r

probing 4074 facts


en:   0%|          | 0/21 [00:00<?, ?it/s]

ja:   0%|          | 0/21 [00:00<?, ?it/s]

## 4. Results

- Number of facts **Unknown in both languages** = the injection set (rule of thumb: if below 300, collect more articles)
- EN × JA cross-tabulation
- Greedy abstention rate (share of "I don't know" / 「わかりません」 answers) per language. A language gap in abstention would confound the evaluation, so it is recorded as a baseline

In [13]:
df = pd.DataFrame([{
    "fact_id": q["fact_id"], "article": q["article"], "type": q["type"],
    "known_en": results[q["fact_id"]]["en"]["label"], "known_ja": results[q["fact_id"]]["ja"]["label"],
    "correct_en": results[q["fact_id"]]["en"]["n_correct"], "correct_ja": results[q["fact_id"]]["ja"]["n_correct"],
    "abstain_en": results[q["fact_id"]]["en"]["abstain_greedy"], "abstain_ja": results[q["fact_id"]]["ja"]["abstain_greedy"],
} for q in qa])
df["inject"] = (df["known_en"] == "Unknown") & (df["known_ja"] == "Unknown")

n_greedy = 2 * N_EX
cross = pd.crosstab(df["known_en"], df["known_ja"], margins=True)
abstain = {lang: round(df[f"abstain_{lang}"].sum() / (n_greedy * len(df)), 3) for lang in ("en", "ja")}
by_type = df.groupby("type")["inject"].agg(["size", "sum", "mean"]).rename(
    columns={"size": "facts", "sum": "inject", "mean": "inject_rate"}).round(3)

print(f"Injection set (Unknown in both languages): {int(df['inject'].sum())} / {len(df)} facts")
print("\nEN (rows) × JA (cols):\n" + cross.to_string())
print(f"\nGreedy abstention rate: en {abstain['en']:.1%}   ja {abstain['ja']:.1%}")
print("\nInjection set by fact type:\n" + by_type.to_string())

Injection set (Unknown in both languages): 2806 / 4074 facts

EN (rows) × JA (cols):
known_ja  Known  Unknown   All
known_en                      
Known       785      274  1059
Unknown     209     2806  3015
All         994     3080  4074

Greedy abstention rate: en 54.0%   ja 4.4%

Injection set by fact type:
             facts  inject  inject_rate
type                                   
categorical   1061     668        0.630
date           717     676        0.943
entity        1917    1224        0.638
numerical      379     238        0.628


In [14]:
def show(fact_id):
    """Show one fact's questions, gold answer and all answers (✓ = scored correct)."""
    q = next(x for x in qa if x["fact_id"] == fact_id)
    print(f"=== {fact_id}  [{q['type']}]  {q['article']}   gold: {q['answer']!r}")
    for lang in ("en", "ja"):
        r = results[fact_id][lang]
        print(f"\n  [{lang}] {r['label']}  correct {r['n_correct']}/{r['n_total']}")
        for question in q[f"q_{lang}"][:2]:
            print(f"    Q: {question}")
        for a in r["greedy"]:
            print(f"    greedy {'✓' if is_correct(a, golds_for(q, lang)) else ' '} {a!r}")
        print(f"    samples (first 8): {r['samples'][:8]}")


# Example: one fact labeled Known and one labeled Unknown
for label in ("Known", "Unknown"):
    hit = df[df["known_en"] == label]
    if len(hit):
        show(hit.iloc[0]["fact_id"])
        print()

=== 83148603-7  [categorical]  1-2 Special   gold: 'film festivals'

  [en] Known  correct 1/72
    Q: What is the primary source of 1-2 Special Inc.'s acquisitions?
    Q: 1-2 Special Inc. mainly acquires films from where?
    greedy   "I don't know"
    greedy   "I don't know"
    greedy   "I don't know"
    greedy   "I don't know"
    greedy   "I don't know"
    greedy   "I don't know"
    greedy   'Japan'
    greedy   "I don't know"
    samples (first 8): ["I don't know", "I don't know", "I don't know", "I don't know", "I don't know", "I don't know", "I don't know", "I don't know"]

  [ja] Unknown  correct 0/72
    Q: 1-2 Special Inc.の主な買付先はどこですか？
    Q: 1-2 Special Inc.は主にどこから作品を買い付けていますか？
    greedy   '1-2 Special Inc.は、主に米国とカナダの小売'
    greedy   'フランス'
    greedy   'フランス'
    greedy   '1-2 Special Inc.は、主に東南アジアの国々から製品'
    greedy   '1-2 Special Inc.は主に日本のアニメ作品を買い付けています。'
    greedy   'イギリス'
    greedy   'イギリス'
    greedy   'イギリス'
    samples (first 8): ['メキシコ', '1-2 Special Inc.の

## 5. Save

In [15]:
with open(OUT_FILE, "w", encoding="utf-8") as out:
    for q in qa:
        row = dict(facts[q["fact_id"]])
        r = results[q["fact_id"]]
        row["known_en"], row["known_ja"] = r["en"]["label"], r["ja"]["label"]
        row["probe"] = r   # per language: {label, n_correct, n_total, greedy, samples, abstain_greedy}
        out.write(json.dumps(row, ensure_ascii=False) + "\n")

summary = {
    "model": model_name, "n_ex": N_EX, "n_samples": N_SAMPLES, "temperature": TEMPERATURE,
    "n_facts": len(df), "n_inject": int(df["inject"].sum()),
    "known": {lang: df[f"known_{lang}"].value_counts().to_dict() for lang in ("en", "ja")},
    "cross_en_ja": pd.crosstab(df["known_en"], df["known_ja"]).to_dict(),
    "abstain_rate_greedy": abstain,
    "by_type": by_type.to_dict("index"),
}
with open(SUMMARY_FILE, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=1)
print(f"wrote {len(qa)} facts → {OUT_FILE}\nsummary → {SUMMARY_FILE}")

wrote 4074 facts → experiment_data/facts_probed.jsonl
summary → experiment_data/probe_summary.json


## 6. Next step

```bash
# Generate paraphrases only for articles that contain injection facts (Unknown in both languages); OpenAI API, can run locally
python scripts/make_paraphrases.py --facts experiment_data/facts_probed.jsonl --unknown-only --n 6 --model gpt-5-mini
```